# Data Analysis Agent
**Hongchao Hu & Niyati Gohil — CS539 Project**

This notebook runs the full Data Analysis Agent inside Google Colab and exposes it as a public web app with no local setup.

### How to use
1. Open this notebook in Google Colab
2. Run Cell 1 to install dependencies
3. Run Cell 2 to enter your Gemini API key in Colab (hidden input)
4. Run all remaining cells
5. Click the public URL shown in the last cell
6. In the web UI, upload a CSV and enter your analysis prompt

### Notes
- No local launcher scripts are needed in Colab
- No local Python virtual environment is needed
- The public URL changes when the runtime restarts

In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import sys

print("Installing dependencies...")
!pip install -q \
    google-generativeai \
    python-dotenv \
    pandas \
    numpy \
    scikit-learn \
    matplotlib \
    seaborn \
    fastapi \
    "uvicorn[standard]" \
    pydantic \
    python-multipart \
    httpx

print("✅ Dependencies installed")

Installing dependencies...
✅ Dependencies installed


Access is denied.


In [ ]:
# Cell 2: Enter Gemini API key (hidden input)
from getpass import getpass
import os

api_key = getpass('Enter your Gemini API key: ').strip()
if not api_key:
    raise ValueError('Gemini API key is required to run analysis.')

os.environ['GEMINI_API_KEY'] = api_key
print('API key loaded into Colab runtime environment.')

In [ ]:
# ── Cell 2: Write project files ───────────────────────────────────────────────
# This cell reconstructs the project source tree directly inside Colab.

import os
from pathlib import Path

ROOT = Path("/content/data_analysis_agent")
ROOT.mkdir(exist_ok=True)
os.chdir(ROOT)

for d in ["src/tools", "static/css", "static/js", "uploads", "artifacts", "outputs"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

# ── src/__init__.py ──
(ROOT / "src/__init__.py").write_text("")
(ROOT / "src/tools/__init__.py").write_text("")

# ── src/config.py ──
(ROOT / "src/config.py").write_text('''
"""Configuration settings for the Data Analysis Agent"""
import os

class Config:
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-1.5-flash")
    TEMPERATURE = float(os.getenv("TEMPERATURE", "0.7"))
    MAX_TOKENS = int(os.getenv("MAX_TOKENS", "2048"))
    MAX_ROWS_TO_DISPLAY = 10
    MAX_COLUMNS_TO_ANALYZE = 50
    CORRELATION_THRESHOLD = 0.5
    FIGURE_SIZE = (10, 6)
    DPI = 100
    OUTPUT_DIR = "outputs"
    SAVE_FIGURES = True
''')

print("✅ Project structure created")

In [ ]:
# ── Cell 3: Write tool source files ───────────────────────────────────────────

(ROOT / "src/tools/inspection.py").write_text(r'''
"""Dataset Inspection Tool"""
import pandas as pd
from typing import Dict, Any, Optional
import os

class DatasetInspectionTool:
    def __init__(self):
        self.df: Optional[pd.DataFrame] = None
        self.file_path: Optional[str] = None

    def load_dataset(self, file_path: str) -> Dict[str, Any]:
        try:
            self.file_path = file_path
            self.df = pd.read_csv(file_path)
            return {"success": True, "message": f"Loaded {file_path}", "basic_info": self.get_basic_info()}
        except Exception as e:
            return {"success": False, "message": str(e), "basic_info": None}

    def get_basic_info(self) -> Dict[str, Any]:
        if self.df is None:
            return {"error": "No dataset loaded"}
        return {
            "file_name": os.path.basename(self.file_path) if self.file_path else "Unknown",
            "num_rows": len(self.df),
            "num_columns": len(self.df.columns),
            "column_names": self.df.columns.tolist(),
            "data_types": self.df.dtypes.astype(str).to_dict(),
            "memory_usage_mb": self.df.memory_usage(deep=True).sum() / 1024**2
        }

    def get_missing_values_info(self) -> Dict[str, Any]:
        if self.df is None:
            return {"error": "No dataset loaded"}
        missing_counts = self.df.isnull().sum()
        missing_percentages = (missing_counts / len(self.df) * 100).round(2)
        missing_info = pd.DataFrame({"missing_count": missing_counts, "missing_percentage": missing_percentages})
        missing_info = missing_info[missing_info["missing_count"] > 0]
        return {
            "total_missing_values": int(missing_counts.sum()),
            "columns_with_missing": missing_info.to_dict("index"),
            "percentage_complete": round((1 - self.df.isnull().sum().sum() / self.df.size) * 100, 2)
        }

    def get_dataframe(self) -> Optional[pd.DataFrame]:
        return self.df
''')

(ROOT / "src/tools/statistics.py").write_text(r'''
"""Statistical Analysis Tool"""
import pandas as pd
import numpy as np
from typing import Dict, Any, List, Optional

class StatisticalAnalysisTool:
    def __init__(self, df=None):
        self.df = df

    def set_dataframe(self, df):
        self.df = df

    def get_descriptive_statistics(self, columns=None):
        if self.df is None:
            return {"error": "No dataframe set"}
        numeric_df = self.df.select_dtypes(include=[np.number])
        if columns:
            numeric_df = numeric_df[columns]
        if numeric_df.empty:
            return {"error": "No numeric columns found"}
        stats = numeric_df.describe().to_dict()
        additional = {}
        for col in numeric_df.columns:
            additional[col] = {
                "variance": float(numeric_df[col].var()),
                "skewness": float(numeric_df[col].skew()),
                "kurtosis": float(numeric_df[col].kurtosis())
            }
        return {"basic_statistics": stats, "additional_statistics": additional}

    def get_categorical_distribution(self, columns=None, top_n=10):
        if self.df is None:
            return {"error": "No dataframe set"}
        categorical_df = self.df.select_dtypes(include=["object", "category"])
        if columns:
            categorical_df = categorical_df[columns]
        if categorical_df.empty:
            return {"error": "No categorical columns found"}
        distributions = {}
        for col in categorical_df.columns:
            vc = self.df[col].value_counts()
            distributions[col] = {
                "unique_values": int(self.df[col].nunique()),
                "top_values": vc.head(top_n).to_dict(),
                "top_percentages": (vc.head(top_n) / len(self.df) * 100).round(2).to_dict()
            }
        return distributions

    def get_correlation_matrix(self, method="pearson", threshold=None):
        if self.df is None:
            return {"error": "No dataframe set"}
        numeric_df = self.df.select_dtypes(include=[np.number])
        if numeric_df.empty or len(numeric_df.columns) < 2:
            return {"error": "Need at least 2 numeric columns"}
        corr = numeric_df.corr(method=method)
        high = []
        for i in range(len(corr.columns)):
            for j in range(i + 1, len(corr.columns)):
                v = corr.iloc[i, j]
                if threshold is None or abs(v) >= threshold:
                    high.append({"variable_1": corr.columns[i], "variable_2": corr.columns[j], "correlation": round(float(v), 3)})
        high.sort(key=lambda x: abs(x["correlation"]), reverse=True)
        return {"correlation_matrix": corr.round(3).to_dict(), "high_correlations": high, "method": method}
''')

print("✅ Tool files written")

In [ ]:
# ── Cell 4: Write visualization tool (Gemini-generated code) ──────────────────

(ROOT / "src/tools/visualization.py").write_text(r'''
"""Visualization Tool — delegates all chart code generation to Gemini."""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
from datetime import datetime
from typing import Dict, Any, List, Optional


class VisualizationTool:
    def __init__(self, df=None, output_dir="outputs", gemini_model=None):
        self.df = df
        self.output_dir = output_dir
        self.gemini_model = gemini_model
        self.created_plots = []
        os.makedirs(output_dir, exist_ok=True)

    def set_dataframe(self, df):
        self.df = df

    # ── helpers ───────────────────────────────────────────────────────────────

    def _dataset_context(self) -> str:
        """Return a concise text description of the loaded dataframe for Gemini."""
        if self.df is None:
            return "No dataset loaded."
        lines = [
            f"Shape: {self.df.shape}",
            f"Columns: {list(self.df.columns)}",
            f"Dtypes:\n{self.df.dtypes.astype(str).to_string()}",
            f"Sample (first 3 rows):\n{self.df.head(3).to_string()}",
        ]
        return "\n".join(lines)

    def _extract_code(self, text: str) -> str:
        """Pull the first Python code block out of a Gemini markdown response."""
        match = re.search(r"```python\n(.*?)```", text, re.DOTALL)
        if match:
            return match.group(1).strip()
        match = re.search(r"```\n(.*?)```", text, re.DOTALL)
        if match:
            return match.group(1).strip()
        return text.strip()

    def _ask_gemini_for_code(self, prompt: str) -> str:
        """Send a prompt to Gemini and return the extracted Python code."""
        if self.gemini_model is None:
            raise RuntimeError("No Gemini model provided to VisualizationTool.")
        response = self.gemini_model.generate_content(prompt)
        return self._extract_code(response.text)

    def _exec_code(self, code: str, save_path: str) -> None:
        """Execute Gemini-generated code with the dataframe and save_path in scope."""
        exec_globals = {
            "df": self.df,
            "save_path": save_path,
            "plt": plt,
            "pd": pd,
            "np": np,
            "sns": sns,
        }
        exec(code, exec_globals)   # noqa: S102
        plt.close("all")
        self.created_plots.append(save_path)

    def _save_path(self, base_name: str) -> str:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        return os.path.join(self.output_dir, f"{base_name}_{ts}.png")

    # ── public API ────────────────────────────────────────────────────────────

    def create_distribution_grid(self, columns=None, save=True) -> Dict[str, Any]:
        if self.df is None:
            return {"error": "No dataframe set"}
        cols = columns or list(self.df.select_dtypes(include=[np.number]).columns)
        save_path = self._save_path("distribution_grid") if save else None

        prompt = f"""
You are a Python data visualization expert.

Dataset information:
{self._dataset_context()}

Task:
Write Python code that creates a grid of histograms showing the distribution
of these numeric columns: {cols}.

Rules:
- The variable `df` (a pandas DataFrame) is already in scope.
- The variable `save_path` (a string file path) is already in scope.
- `plt`, `pd`, `np`, `sns` are already imported and in scope.
- Save the final figure with: plt.savefig(save_path, dpi=100, bbox_inches="tight")
- Do NOT call plt.show().
- Use a sensible figure size and plt.tight_layout().
- Output ONLY a single Python code block. No explanations, no prose.
"""
        code = self._ask_gemini_for_code(prompt)
        self._exec_code(code, save_path)
        return {"type": "distribution_grid", "columns": cols,
                "file_path": save_path, "generated_code": code}

    def create_correlation_heatmap(self, columns=None, method="pearson",
                                    title=None, save=True) -> Dict[str, Any]:
        if self.df is None:
            return {"error": "No dataframe set"}
        cols = columns or list(self.df.select_dtypes(include=[np.number]).columns)
        save_path = self._save_path("correlation_heatmap") if save else None
        chart_title = title or f"Correlation Heatmap ({method})"

        prompt = f"""
You are a Python data visualization expert.

Dataset information:
{self._dataset_context()}

Task:
Write Python code that creates a seaborn correlation heatmap for these numeric
columns: {cols}. Use method="{method}", annotate cells, cmap="coolwarm", center=0.

Rules:
- The variable `df` (a pandas DataFrame) is already in scope.
- The variable `save_path` (a string file path) is already in scope.
- `plt`, `pd`, `np`, `sns` are already imported and in scope.
- Set the chart title to: "{chart_title}"
- Save the final figure with: plt.savefig(save_path, dpi=100, bbox_inches="tight")
- Do NOT call plt.show().
- Output ONLY a single Python code block. No explanations, no prose.
"""
        code = self._ask_gemini_for_code(prompt)
        self._exec_code(code, save_path)
        return {"type": "correlation_heatmap", "method": method,
                "file_path": save_path, "generated_code": code}

    def create_box_plot(self, columns=None, title=None, save=True) -> Dict[str, Any]:
        if self.df is None:
            return {"error": "No dataframe set"}
        cols = columns or list(self.df.select_dtypes(include=[np.number]).columns)
        save_path = self._save_path("box_plot") if save else None
        chart_title = title or "Box Plot - Outlier Detection"

        prompt = f"""
You are a Python data visualization expert.

Dataset information:
{self._dataset_context()}

Task:
Write Python code that creates a box plot for these numeric columns: {cols}.
Use matplotlib or seaborn. Rotate x-axis labels if needed.

Rules:
- The variable `df` (a pandas DataFrame) is already in scope.
- The variable `save_path` (a string file path) is already in scope.
- `plt`, `pd`, `np`, `sns` are already imported and in scope.
- Set the chart title to: "{chart_title}"
- Save the final figure with: plt.savefig(save_path, dpi=100, bbox_inches="tight")
- Do NOT call plt.show().
- Output ONLY a single Python code block. No explanations, no prose.
"""
        code = self._ask_gemini_for_code(prompt)
        self._exec_code(code, save_path)
        return {"type": "box_plot", "columns": cols,
                "file_path": save_path, "generated_code": code}

    def create_bar_chart(self, column, top_n=10, title=None, save=True) -> Dict[str, Any]:
        if self.df is None:
            return {"error": "No dataframe set"}
        save_path = self._save_path(f"bar_chart_{column}") if save else None
        chart_title = title or f"Top {top_n} Categories in {column}"

        prompt = f"""
You are a Python data visualization expert.

Dataset information:
{self._dataset_context()}

Task:
Write Python code that creates a bar chart showing the top {top_n} value counts
for the column "{column}". Display count labels on top of each bar.

Rules:
- The variable `df` (a pandas DataFrame) is already in scope.
- The variable `save_path` (a string file path) is already in scope.
- `plt`, `pd`, `np`, `sns` are already imported and in scope.
- Set the chart title to: "{chart_title}"
- Save the final figure with: plt.savefig(save_path, dpi=100, bbox_inches="tight")
- Do NOT call plt.show().
- Output ONLY a single Python code block. No explanations, no prose.
"""
        code = self._ask_gemini_for_code(prompt)
        self._exec_code(code, save_path)
        return {"type": "bar_chart", "column": column,
                "file_path": save_path, "generated_code": code}

    def get_created_plots(self) -> List[str]:
        return self.created_plots
''')

print("✅ Visualization tool written (Gemini-powered)")

In [ ]:
# ── Cell 5: Write agent ────────────────────────────────────────────────────────

(ROOT / "src/agent.py").write_text(r'''
"""Data Analysis Agent"""
import json, traceback
from typing import Dict, Any, Optional
import google.generativeai as genai
from .config import Config
from .tools.inspection import DatasetInspectionTool
from .tools.statistics import StatisticalAnalysisTool
from .tools.visualization import VisualizationTool

class DataAnalysisAgent:
    def __init__(self, api_key: Optional[str] = None):
        self.api_key = api_key or Config.GEMINI_API_KEY
        genai.configure(api_key=self.api_key)
        self.model = genai.GenerativeModel(Config.GEMINI_MODEL)
        self.inspection_tool = DatasetInspectionTool()
        self.statistics_tool = StatisticalAnalysisTool()
        self.visualization_tool = VisualizationTool()

    def analyze(self, file_path: str, user_prompt: str) -> Dict[str, Any]:
        results = {"user_prompt": user_prompt, "file_path": file_path, "steps": [], "visualizations": [], "summary": "", "success": True, "execution_steps": []}
        try:
            load_result = self.inspection_tool.load_dataset(file_path)
            if not load_result["success"]:
                results["success"] = False
                results["error"] = load_result["message"]
                return results
            df = self.inspection_tool.get_dataframe()
            basic_info = load_result["basic_info"]
            results["steps"].append({"step": "load_dataset", "result": basic_info})
            self.statistics_tool.set_dataframe(df)
            self.visualization_tool.set_dataframe(df)
            analysis_code = self._generate_analysis_code(basic_info, user_prompt)
            if not analysis_code:
                results["success"] = False
                results["error"] = "Failed to generate analysis code"
                return results
            results["execution_steps"].append("Code generation completed")
            execution_results = self._execute_analysis_code(analysis_code, df, basic_info, user_prompt)
            if not execution_results["success"]:
                results["success"] = False
                results["error"] = execution_results.get("error", "Code execution failed")
                results["execution_steps"] = execution_results.get("execution_steps", [])
                return results
            results["visualizations"].extend(execution_results.get("visualizations", []))
            results["steps"].extend(execution_results.get("analysis_results", []))
            results["execution_steps"] = execution_results.get("execution_steps", [])
            execution_summary = execution_results.get("summary", "")
            summary = self._generate_summary(user_prompt, basic_info, execution_results, execution_summary)
            results["summary"] = summary
        except Exception as e:
            results["success"] = False
            results["error"] = f"Analysis failed: {str(e)}"
            results["error_traceback"] = traceback.format_exc()
        return results

    def _generate_analysis_code(self, basic_info: Dict[str, Any], user_prompt: str) -> Optional[str]:
        tools_available = """
# Available Tools:
# - inspection_tool: DatasetInspectionTool for data loading/inspection
# - statistics_tool: StatisticalAnalysisTool with methods:
#   - get_descriptive_statistics()
#   - get_correlation_matrix(threshold=0.3)
#   - get_categorical_distribution()
#   - get_numerical_distribution()
#   - get_outlier_detection()
# - visualization_tool: VisualizationTool with methods:
#   - create_distribution_grid()
#   - create_correlation_heatmap()
#   - create_box_plot()
#   - create_scatter_plot()
#   - create_categorical_plot()
# - df: pandas DataFrame with the loaded data
"""
        prompt = f"""You are a Python data analysis expert. Generate executable Python code to analyze a dataset based on the user's request.

{tools_available}

Dataset: Rows={basic_info['num_rows']}, Columns={basic_info['num_columns']}
Columns: {basic_info['column_names']}
Types: {json.dumps(basic_info['data_types'], indent=2)}

User: {user_prompt}

Generate ONLY valid Python code that:
1. Uses available tools (inspection_tool, statistics_tool, visualization_tool)
2. Analyzes data per user request
3. Defines analysis_results dict with keys: visualizations (list), analysis_steps (list), summary (str)
4. Stores visualization file paths in analysis_results['visualizations']
5. Stores analysis steps (dicts with analysis/result) in analysis_results['analysis_steps']
6. Writes 2-3 sentence summary to analysis_results['summary']

Example structure:
analysis_results = {{'visualizations': [], 'analysis_steps': [], 'summary': ''}}
# Your analysis code...
analysis_results['analysis_steps'].append({{'analysis': 'name', 'result': data}})
plot = visualization_tool.create_correlation_heatmap()
if plot and 'file_path' in plot: analysis_results['visualizations'].append(plot['file_path'])
analysis_results['summary'] = 'Summary text...'
"""
        try:
            response = self.model.generate_content(prompt)
            code = response.text.strip()
            if code.startswith("```python"): code = code[9:]
            elif code.startswith("```"): code = code[3:]
            if code.endswith("```"): code = code[:-3]
            return code.strip()
        except: return None

    def _execute_analysis_code(self, code: str, df, basic_info: Dict[str, Any], user_prompt: str) -> Dict[str, Any]:
        execution_results = {"success": True, "visualizations": [], "analysis_results": [], "summary": "", "execution_steps": [], "error": None}
        try:
            execution_results["execution_steps"].append("Starting code execution...")
            execution_env = {
                'df': df,
                'inspection_tool': self.inspection_tool,
                'statistics_tool': self.statistics_tool,
                'visualization_tool': self.visualization_tool,
                'basic_info': basic_info,
                'json': json,
                '__builtins__': {'print': lambda *a, **k: None, 'len': len, 'range': range, 'str': str, 'int': int, 'float': float, 'list': list, 'dict': dict, 'tuple': tuple, 'set': set, 'Exception': Exception}
            }
            exec(code, execution_env)
            execution_results["execution_steps"].append("Code executed successfully")
            if 'analysis_results' in execution_env:
                ar = execution_env['analysis_results']
                if isinstance(ar.get('visualizations'), list):
                    execution_results["visualizations"] = ar['visualizations']
                    execution_results["execution_steps"].append(f"Generated {len(ar['visualizations'])} visualization(s)")
                if isinstance(ar.get('analysis_steps'), list):
                    for step in ar['analysis_steps']:
                        execution_results["analysis_results"].append(step)
                    execution_results["execution_steps"].append(f"Completed {len(ar['analysis_steps'])} analysis step(s)")
                if isinstance(ar.get('summary'), str):
                    execution_results["summary"] = ar['summary']
                    execution_results["execution_steps"].append("Generated analysis summary")
            execution_results["execution_steps"].append("Execution completed")
        except Exception as e:
            execution_results["success"] = False
            execution_results["error"] = f"Code execution failed: {str(e)}"
            execution_results["execution_steps"].append(f"Error: {str(e)}")
        return execution_results

    def _generate_summary(self, user_prompt: str, basic_info: Dict[str, Any], execution_results: Dict[str, Any], execution_summary: str = "") -> str:
        if execution_summary:
            prompt = f"""You are a data scientist. Refine and enhance this analysis summary:

User Question: {user_prompt}
Dataset: {basic_info['num_rows']} rows, {basic_info['num_columns']} columns

Summary from Analysis:
{execution_summary}

Analysis Steps:
{json.dumps(execution_results.get('analysis_results', []), indent=2, default=str)}

Enhance this summary to be more engaging, insightful, and actionable. Answer the user's question specifically."""
            try:
                return self.model.generate_content(prompt).text
            except:
                return execution_summary if execution_summary else "Analysis completed."
        data_types_summary = {}
        for col, dtype in basic_info['data_types'].items():
            if dtype not in data_types_summary: data_types_summary[dtype] = []
            data_types_summary[dtype].append(col)
        viz_count = len(execution_results.get('visualizations', []))
        analysis_count = len(execution_results.get('analysis_results', []))
        return f"""## Analysis Summary\n\n**Dataset Overview**\n- Records: {basic_info['num_rows']:,}\n- Features: {basic_info['num_columns']}\n\n**Analysis Results**\n- Analyses: {analysis_count}\n- Visualizations: {viz_count}\n\n**Your Question**\n{user_prompt}\n\nReview the visualizations above for detailed insights."""

    def quick_analyze(self, file_path: str) -> Dict[str, Any]:
        prompt = """Perform exploratory data analysis. Generate visualizations for: 1) Distribution of numeric variables 2) Relationships between variables 3) Any patterns or outliers. Provide key findings summary."""
        return self.analyze(file_path, prompt)
''')

print("✅ Agent written")

In [ ]:
# ── Cell 6: Write FastAPI app ──────────────────────────────────────────────────

(ROOT / "app.py").write_text(r'''
"""FastAPI Web Application for Data Analysis Agent"""
from fastapi import FastAPI, HTTPException, UploadFile, File, Form, BackgroundTasks, Header
from fastapi.responses import JSONResponse, FileResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any
import os, json, time, uuid, shutil
from pathlib import Path

from src.agent import DataAnalysisAgent
from src.config import Config

UPLOAD_DIR = Path("uploads")
ARTIFACTS_DIR = Path("artifacts")
OUTPUT_DIR = Path("outputs")
STATIC_DIR = Path("static")

for d in [UPLOAD_DIR, ARTIFACTS_DIR, OUTPUT_DIR, STATIC_DIR]:
    d.mkdir(exist_ok=True)


class AnalysisResponse(BaseModel):
    artifact_id: str
    query: str
    success: bool
    summary: str
    analysis_plan: List[Dict[str, Any]]
    steps_executed: int
    visualizations: List[str]
    evaluation: Optional[Dict[str, Any]] = None
    latency: float
    artifact_path: Optional[str] = None


def create_app() -> FastAPI:
    app = FastAPI(title="Data Analysis Agent API", version="1.0.0", docs_url="/docs")
    app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])
    if STATIC_DIR.exists():
        app.mount("/static", StaticFiles(directory=str(STATIC_DIR)), name="static")
    return app


app = create_app()


def _get_agent(api_key: Optional[str]) -> DataAnalysisAgent:
    """Create a per-request agent using the caller-supplied API key."""
    key = api_key or Config.GEMINI_API_KEY
    if not key:
        raise HTTPException(status_code=401, detail="Gemini API key is required. Pass it in the X-API-Key header.")
    return DataAnalysisAgent(api_key=key)


def save_artifact(analysis_id, results):
    path = ARTIFACTS_DIR / f"{analysis_id}.json"
    path.write_text(json.dumps(results, indent=2, default=str))
    return str(path)


def cleanup_old(directory: Path, max_age_hours=24):
    import time as _time
    now = _time.time()
    for p in directory.glob("*"):
        if p.is_file() and (now - p.stat().st_mtime) / 3600 > max_age_hours:
            p.unlink()


@app.get("/")
async def root():
    idx = STATIC_DIR / "index.html"
    if idx.exists():
        return FileResponse(idx)
    return {"service": "Data Analysis Agent API", "docs": "/docs"}


@app.get("/health")
async def health():
    return {"status": "healthy", "version": "1.0.0", "agent_initialized": True}


@app.post("/upload-analyze", response_model=AnalysisResponse)
async def upload_and_analyze(
    file: UploadFile = File(...),
    query: str = Form(...),
    background_tasks: BackgroundTasks = None,
    x_api_key: Optional[str] = Header(None)
):
    if not file.filename.endswith(".csv"):
        raise HTTPException(status_code=400, detail="Only CSV files are supported")

    agent = _get_agent(x_api_key)

    upload_id = str(uuid.uuid4())
    file_path = UPLOAD_DIR / f"{upload_id}_{file.filename}"
    try:
        with open(file_path, "wb") as buf:
            shutil.copyfileobj(file.file, buf)

        start = time.time()
        results = agent.analyze(str(file_path), query)
        latency = round(time.time() - start, 2)

        if not results.get("success"):
            raise HTTPException(status_code=500, detail=results.get("error", "Analysis failed"))

        analysis_id = str(uuid.uuid4())
        artifact_path = save_artifact(analysis_id, results)

        if background_tasks:
            background_tasks.add_task(cleanup_old, ARTIFACTS_DIR, 24)
            background_tasks.add_task(cleanup_old, OUTPUT_DIR, 24)

        viz_files = []
        for viz in results.get("visualizations", []):
            if isinstance(viz, dict) and "file_path" in viz:
                fp = viz["file_path"]
                viz_files.append(fp.split("/")[-1] if "/" in fp else fp.split("\\")[-1])
            elif isinstance(viz, str):
                viz_files.append(viz)

        return AnalysisResponse(
            artifact_id=analysis_id,
            query=query,
            success=True,
            summary=results["summary"],
            analysis_plan=results.get("analysis_plan", []),
            steps_executed=len(results.get("steps", [])),
            visualizations=viz_files,
            evaluation=None,
            latency=latency,
            artifact_path=artifact_path
        )
    except HTTPException:
        raise
    except Exception as e:
        if file_path.exists():
            file_path.unlink()
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/artifact/{analysis_id}")
async def get_artifact(analysis_id: str):
    path = ARTIFACTS_DIR / f"{analysis_id}.json"
    if not path.exists():
        raise HTTPException(status_code=404, detail="Artifact not found")
    return FileResponse(path, media_type="application/json")


@app.get("/visualization/{filename}")
async def get_visualization(filename: str):
    # Security: prevent path traversal
    safe = Path(filename).name
    path = OUTPUT_DIR / safe
    if not path.exists():
        raise HTTPException(status_code=404, detail="Visualization not found")
    return FileResponse(path, media_type="image/png")


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
''')

print("✅ app.py written")

In [ ]:
# ── Cell 7: Write frontend files (original UI) ─────────────────────────────────

html = r'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Data Analysis Agent - Upload & Analyze</title>
    <link rel="stylesheet" href="/static/css/styles.css">
</head>
<body>
    <div class="container">
        <header class="header">
            <h1>📊 Data Analysis Agent</h1>
            <p class="subtitle">Drop your CSV file and get instant insights powered by AI</p>
            <div class="status-indicator">
                <span class="status-dot" id="statusDot"></span>
                <span id="statusText">Checking status...</span>
            </div>
        </header>

        <main class="main-content">
            <section class="upload-section" id="uploadSection">
                <div class="upload-area" id="uploadArea">
                    <div class="upload-icon">📁</div>
                    <h2>Drop your CSV file here</h2>
                    <p>or click to browse</p>
                    <input type="file" id="fileInput" accept=".csv" hidden>
                    <div class="file-info" id="fileInfo" style="display: none;">
                        <span class="file-name" id="fileName"></span>
                        <button class="btn-remove" id="removeFile">✕</button>
                    </div>
                </div>

                <div class="query-section">
                    <label for="queryInput">What would you like to know?</label>
                    <textarea
                        id="queryInput"
                        placeholder="e.g., What is the distribution of sales by region? What patterns exist in this data?"
                        rows="3"
                    ></textarea>

                    <div class="example-queries">
                        <p>Example questions:</p>
                        <div class="query-chips">
                            <button class="query-chip" data-query="Provide a comprehensive exploratory data analysis">📊 Full EDA</button>
                            <button class="query-chip" data-query="What are the key patterns and correlations in this data?">🔍 Find Patterns</button>
                            <button class="query-chip" data-query="Show me the distribution of values and identify any outliers">📈 Distributions</button>
                            <button class="query-chip" data-query="Compare different categories and identify top performers">🏆 Compare Categories</button>
                        </div>
                    </div>
                </div>

                <button class="btn-primary" id="analyzeBtn" disabled>
                    <span class="btn-text">Analyze Dataset</span>
                    <span class="btn-loading" style="display: none;">
                        <span class="spinner"></span> Analyzing...
                    </span>
                </button>
            </section>

            <section class="results-section" id="resultsSection" style="display: none;">
                <div class="results-header">
                    <h2>Analysis Results</h2>
                    <button class="btn-secondary" id="newAnalysisBtn">📤 New Analysis</button>
                </div>

                <div class="results-grid">
                    <div class="result-card summary-card">
                        <div class="card-header">
                            <h3>💡 Summary</h3>
                            <span class="quality-badge" id="qualityBadge"></span>
                        </div>
                        <div class="card-content">
                            <div id="summaryText" class="summary-text"></div>
                            <div class="metrics">
                                <div class="metric">
                                    <span class="metric-label">Analysis Time</span>
                                    <span class="metric-value" id="latencyValue">-</span>
                                </div>
                                <div class="metric">
                                    <span class="metric-label">Steps Executed</span>
                                    <span class="metric-value" id="stepsValue">-</span>
                                </div>
                                <div class="metric">
                                    <span class="metric-label">Visualizations</span>
                                    <span class="metric-value" id="vizValue">-</span>
                                </div>
                            </div>
                        </div>
                    </div>

                    <div class="result-card">
                        <div class="card-header">
                            <h3>📊 Visualizations</h3>
                        </div>
                        <div class="card-content">
                            <div id="visualizationsContainer" class="visualizations-grid"></div>
                        </div>
                    </div>

                    <div class="result-card">
                        <div class="card-header">
                            <h3>🔍 Analysis Steps</h3>
                        </div>
                        <div class="card-content">
                            <div id="stepsContainer" class="steps-list"></div>
                        </div>
                    </div>

                    <div class="result-card" id="evaluationCard" style="display: none;">
                        <div class="card-header">
                            <h3>⭐ Quality Evaluation</h3>
                        </div>
                        <div class="card-content">
                            <div id="evaluationContent" class="evaluation-content"></div>
                        </div>
                    </div>
                </div>

                <div class="download-section">
                    <h3>📥 Download Results</h3>
                    <div class="download-buttons">
                        <button class="btn-download" id="downloadArtifactBtn">
                            <span>📄 Full Report (JSON)</span>
                        </button>
                        <button class="btn-download" id="downloadVisualizationsBtn">
                            <span>🖼️ All Visualizations</span>
                        </button>
                    </div>
                </div>
            </section>

            <section class="error-section" id="errorSection" style="display: none;">
                <div class="error-card">
                    <div class="error-icon">⚠️</div>
                    <h3>Analysis Failed</h3>
                    <p id="errorMessage"></p>
                    <button class="btn-secondary" id="tryAgainBtn">Try Again</button>
                </div>
            </section>
        </main>

        <footer class="footer">
            <p>Data Analysis Agent</p>
            <p>Hongchao Hu & Niyati Gohil</p>
        </footer>
    </div>

    <script src="/static/js/app.js"></script>
</body>
</html>'''

(ROOT / "static/index.html").write_text(html)
print("✅ index.html written")

In [ ]:
# ── Cell 8: Keep original CSS from Cell 10 output ─────────────────────────────
print("Cell 8 skipped: using original CSS written in Cell 10.")

In [ ]:
# ── Cell 9: Write JavaScript (original behavior) ──────────────────────────────

js = r'''
// ==================== Configuration ====================
const API_BASE_URL = window.location.origin;

// ==================== State Management ====================
let selectedFile = null;
let currentArtifactId = null;
let currentVisualizations = [];

// ==================== DOM Elements ====================
const elements = {
    uploadArea: document.getElementById('uploadArea'),
    fileInput: document.getElementById('fileInput'),
    fileInfo: document.getElementById('fileInfo'),
    fileName: document.getElementById('fileName'),
    removeFile: document.getElementById('removeFile'),
    queryInput: document.getElementById('queryInput'),
    analyzeBtn: document.getElementById('analyzeBtn'),
    uploadSection: document.getElementById('uploadSection'),
    resultsSection: document.getElementById('resultsSection'),
    errorSection: document.getElementById('errorSection'),
    errorMessage: document.getElementById('errorMessage'),
    statusDot: document.getElementById('statusDot'),
    statusText: document.getElementById('statusText'),
    newAnalysisBtn: document.getElementById('newAnalysisBtn'),
    tryAgainBtn: document.getElementById('tryAgainBtn'),
    summaryText: document.getElementById('summaryText'),
    latencyValue: document.getElementById('latencyValue'),
    stepsValue: document.getElementById('stepsValue'),
    vizValue: document.getElementById('vizValue'),
    qualityBadge: document.getElementById('qualityBadge'),
    visualizationsContainer: document.getElementById('visualizationsContainer'),
    stepsContainer: document.getElementById('stepsContainer'),
    evaluationCard: document.getElementById('evaluationCard'),
    evaluationContent: document.getElementById('evaluationContent'),
    downloadArtifactBtn: document.getElementById('downloadArtifactBtn'),
    downloadVisualizationsBtn: document.getElementById('downloadVisualizationsBtn')
};

// ==================== Initialization ====================
document.addEventListener('DOMContentLoaded', () => {
    checkHealth();
    setupEventListeners();
    elements.queryInput.value = "Provide a comprehensive exploratory data analysis with key insights, patterns, and visualizations.";
});

// ==================== Health Check ====================
async function checkHealth() {
    try {
        const response = await fetch(`${API_BASE_URL}/health`);
        const data = await response.json();

        if (data.status === 'healthy') {
            elements.statusDot.classList.add('healthy');
            elements.statusText.textContent = 'API Ready';
        } else {
            elements.statusText.textContent = 'API Unavailable';
        }
    } catch (error) {
        elements.statusText.textContent = 'Connection Error';
        console.error('Health check failed:', error);
    }
}

// ==================== Event Listeners ====================
function setupEventListeners() {
    elements.uploadArea.addEventListener('click', (e) => {
        if (!e.target.closest('.btn-remove')) {
            elements.fileInput.click();
        }
    });

    elements.fileInput.addEventListener('change', handleFileSelect);

    elements.removeFile.addEventListener('click', (e) => {
        e.stopPropagation();
        clearSelectedFile();
    });

    ['dragenter', 'dragover', 'dragleave', 'drop'].forEach(eventName => {
        elements.uploadArea.addEventListener(eventName, preventDefaults, false);
    });

    ['dragenter', 'dragover'].forEach(eventName => {
        elements.uploadArea.addEventListener(eventName, () => {
            elements.uploadArea.classList.add('dragover');
        });
    });

    ['dragleave', 'drop'].forEach(eventName => {
        elements.uploadArea.addEventListener(eventName, () => {
            elements.uploadArea.classList.remove('dragover');
        });
    });

    elements.uploadArea.addEventListener('drop', handleDrop);
    elements.analyzeBtn.addEventListener('click', handleAnalyze);

    document.querySelectorAll('.query-chip').forEach(chip => {
        chip.addEventListener('click', () => {
            elements.queryInput.value = chip.dataset.query;
        });
    });

    elements.newAnalysisBtn.addEventListener('click', resetToUpload);
    elements.tryAgainBtn.addEventListener('click', resetToUpload);
    elements.downloadArtifactBtn.addEventListener('click', downloadArtifact);
    elements.downloadVisualizationsBtn.addEventListener('click', downloadAllVisualizations);
}

// ==================== File Handling ====================
function preventDefaults(e) {
    e.preventDefault();
    e.stopPropagation();
}

function handleDrop(e) {
    const dt = e.dataTransfer;
    const files = dt.files;

    if (files.length > 0) {
        handleFile(files[0]);
    }
}

function handleFileSelect(e) {
    const files = e.target.files;
    if (files.length > 0) {
        handleFile(files[0]);
    }
}

function handleFile(file) {
    if (!file.name.toLowerCase().endsWith('.csv')) {
        showError('Please upload a CSV file only.');
        return;
    }

    const maxSize = 50 * 1024 * 1024;
    if (file.size > maxSize) {
        showError('File size exceeds 50MB limit.');
        return;
    }

    selectedFile = file;
    elements.fileName.textContent = file.name;
    elements.fileInfo.style.display = 'flex';
    elements.analyzeBtn.disabled = false;

    elements.uploadArea.querySelector('h2').textContent = 'File selected!';
    elements.uploadArea.querySelector('p').textContent = 'Click "Analyze Dataset" to continue';
}

function clearSelectedFile() {
    selectedFile = null;
    elements.fileInput.value = '';
    elements.fileInfo.style.display = 'none';
    elements.analyzeBtn.disabled = true;
    elements.uploadArea.querySelector('h2').textContent = 'Drop your CSV file here';
    elements.uploadArea.querySelector('p').textContent = 'or click to browse';
}

// ==================== Analysis ====================
async function handleAnalyze() {
    if (!selectedFile) {
        showError('Please select a file first.');
        return;
    }

    const query = elements.queryInput.value.trim();
    if (!query) {
        showError('Please enter a query.');
        return;
    }

    elements.analyzeBtn.disabled = true;
    elements.analyzeBtn.querySelector('.btn-text').style.display = 'none';
    elements.analyzeBtn.querySelector('.btn-loading').style.display = 'flex';

    try {
        const formData = new FormData();
        formData.append('file', selectedFile);
        formData.append('query', query);

        const response = await fetch(`${API_BASE_URL}/upload-analyze`, {
            method: 'POST',
            body: formData
        });

        if (!response.ok) {
            const errorData = await response.json();
            throw new Error(errorData.detail || 'Analysis failed');
        }

        const result = await response.json();
        displayResults(result);

    } catch (error) {
        console.error('Analysis error:', error);
        showError(error.message || 'An error occurred during analysis. Please try again.');
        elements.analyzeBtn.disabled = false;
        elements.analyzeBtn.querySelector('.btn-text').style.display = 'inline';
        elements.analyzeBtn.querySelector('.btn-loading').style.display = 'none';
    }
}

// ==================== Results Display ====================
function displayResults(data) {
    elements.uploadSection.style.display = 'none';
    elements.errorSection.style.display = 'none';
    elements.resultsSection.style.display = 'block';

    currentArtifactId = data.artifact_id;
    currentVisualizations = data.visualizations || [];

    elements.summaryText.textContent = data.summary;

    elements.latencyValue.textContent = `${data.latency.toFixed(2)}s`;
    elements.stepsValue.textContent = data.steps_executed || 'N/A';
    elements.vizValue.textContent = currentVisualizations.length;

    if (data.evaluation && data.evaluation.overall_score !== undefined) {
        const score = data.evaluation.overall_score;
        let badgeClass = 'acceptable';
        let badgeText = 'Acceptable';

        if (score >= 0.8) {
            badgeClass = 'excellent';
            badgeText = 'Excellent';
        } else if (score >= 0.6) {
            badgeClass = 'good';
            badgeText = 'Good';
        }

        elements.qualityBadge.className = `quality-badge ${badgeClass}`;
        elements.qualityBadge.textContent = `${badgeText} (${(score * 100).toFixed(0)}%)`;
    } else {
        elements.qualityBadge.style.display = 'none';
    }

    displayVisualizations(currentVisualizations);
    displayAnalysisSteps(data.analysis_plan || []);

    if (data.evaluation) {
        displayEvaluation(data.evaluation);
    }

    elements.analyzeBtn.disabled = false;
    elements.analyzeBtn.querySelector('.btn-text').style.display = 'inline';
    elements.analyzeBtn.querySelector('.btn-loading').style.display = 'none';
}

function displayVisualizations(visualizations) {
    elements.visualizationsContainer.innerHTML = '';

    if (visualizations.length === 0) {
        elements.visualizationsContainer.innerHTML = '<p style="color: var(--text-secondary);">No visualizations generated.</p>';
        return;
    }

    visualizations.forEach((viz, index) => {
        const vizItem = document.createElement('div');
        vizItem.className = 'viz-item';

        const img = document.createElement('img');
        img.src = `${API_BASE_URL}/visualization/${viz}`;
        img.alt = `Visualization ${index + 1}`;
        img.onerror = () => {
            img.src = 'data:image/svg+xml,<svg xmlns="http://www.w3.org/2000/svg" width="400" height="300"><rect width="400" height="300" fill="%23f3f4f6"/><text x="50%" y="50%" text-anchor="middle" fill="%236b7280">Image not available</text></svg>';
        };

        const label = document.createElement('div');
        label.className = 'viz-label';
        label.textContent = formatVisualizationName(viz);

        vizItem.appendChild(img);
        vizItem.appendChild(label);
        elements.visualizationsContainer.appendChild(vizItem);
    });
}

function displayAnalysisSteps(steps) {
    elements.stepsContainer.innerHTML = '';

    if (steps.length === 0) {
        elements.stepsContainer.innerHTML = '<p style="color: var(--text-secondary);">No analysis steps recorded.</p>';
        return;
    }

    steps.forEach((step, index) => {
        const stepItem = document.createElement('div');
        stepItem.className = 'step-item';

        const stepTitle = document.createElement('h4');
        stepTitle.textContent = `${index + 1}. ${step.tool || 'Unknown Tool'}`;

        const stepDesc = document.createElement('p');
        stepDesc.textContent = step.purpose || 'No description available';

        stepItem.appendChild(stepTitle);
        stepItem.appendChild(stepDesc);
        elements.stepsContainer.appendChild(stepItem);
    });
}

function displayEvaluation(evaluation) {
    if (!evaluation || Object.keys(evaluation).length === 0) {
        elements.evaluationCard.style.display = 'none';
        return;
    }

    elements.evaluationCard.style.display = 'block';
    elements.evaluationContent.innerHTML = '';

    if (evaluation.overall_score !== undefined) {
        const overallDiv = document.createElement('div');
        overallDiv.className = 'metric';
        overallDiv.innerHTML = `
            <span class="metric-label">Overall Score</span>
            <span class="metric-value">${(evaluation.overall_score * 100).toFixed(0)}%</span>
        `;
        elements.evaluationContent.appendChild(overallDiv);
    }

    const scores = {
        'Tool Invocation': evaluation.tool_invocation_score,
        'Summary Accuracy': evaluation.summary_accuracy_score,
        'Visualization Quality': evaluation.visualization_quality_score
    };

    Object.entries(scores).forEach(([label, score]) => {
        if (score !== undefined) {
            const scoreDiv = document.createElement('div');
            scoreDiv.className = 'metric';
            scoreDiv.innerHTML = `
                <span class="metric-label">${label}</span>
                <span class="metric-value">${(score * 100).toFixed(0)}%</span>
            `;
            elements.evaluationContent.appendChild(scoreDiv);
        }
    });

    if (evaluation.feedback && evaluation.feedback.length > 0) {
        const feedbackDiv = document.createElement('div');
        feedbackDiv.style.marginTop = '20px';
        feedbackDiv.innerHTML = '<h4 style="margin-bottom: 10px;">Feedback:</h4>';

        const feedbackList = document.createElement('ul');
        feedbackList.style.paddingLeft = '20px';
        evaluation.feedback.forEach(item => {
            const li = document.createElement('li');
            li.textContent = item;
            li.style.marginBottom = '5px';
            feedbackList.appendChild(li);
        });

        feedbackDiv.appendChild(feedbackList);
        elements.evaluationContent.appendChild(feedbackDiv);
    }
}

// ==================== Download Functions ====================
async function downloadArtifact() {
    if (!currentArtifactId) return;

    try {
        const response = await fetch(`${API_BASE_URL}/artifact/${currentArtifactId}`);
        const data = await response.json();

        const blob = new Blob([JSON.stringify(data, null, 2)], { type: 'application/json' });
        const url = window.URL.createObjectURL(blob);
        const a = document.createElement('a');
        a.href = url;
        a.download = `analysis_${currentArtifactId}.json`;
        document.body.appendChild(a);
        a.click();
        window.URL.revokeObjectURL(url);
        document.body.removeChild(a);
    } catch (error) {
        console.error('Download failed:', error);
        alert('Failed to download artifact');
    }
}

async function downloadAllVisualizations() {
    if (currentVisualizations.length === 0) return;

    for (const viz of currentVisualizations) {
        try {
            const response = await fetch(`${API_BASE_URL}/visualization/${viz}`);
            const blob = await response.blob();
            const url = window.URL.createObjectURL(blob);
            const a = document.createElement('a');
            a.href = url;
            a.download = viz;
            document.body.appendChild(a);
            a.click();
            window.URL.revokeObjectURL(url);
            document.body.removeChild(a);
            await new Promise(resolve => setTimeout(resolve, 200));
        } catch (error) {
            console.error(`Failed to download ${viz}:`, error);
        }
    }
}

// ==================== Error Handling ====================
function showError(message) {
    elements.uploadSection.style.display = 'none';
    elements.resultsSection.style.display = 'none';
    elements.errorSection.style.display = 'block';
    elements.errorMessage.textContent = message;
}

// ==================== Navigation ====================
function resetToUpload() {
    clearSelectedFile();
    elements.uploadSection.style.display = 'block';
    elements.resultsSection.style.display = 'none';
    elements.errorSection.style.display = 'none';
    currentArtifactId = null;
    currentVisualizations = [];
}

// ==================== Utility Functions ====================
function formatVisualizationName(filename) {
    let name = filename.replace(/\.(png|jpg|jpeg|svg)$/i, '');
    name = name.replace(/_\d{8}_\d{6}$/, '');
    name = name.replace(/_/g, ' ');
    name = name.charAt(0).toUpperCase() + name.slice(1);
    return name;
}
'''

(ROOT / "static/js/app.js").write_text(js)
print("✅ app.js written")

In [ ]:
# ── Cell 10: Copy CSS from original styles (base styles) ──────────────────────
# This cell writes the base CSS that styles.css.backup may be missing.
# We write a full standalone CSS so the Colab version is self-contained.

base_css = r"""
:root {
    --primary-color: #333333;
    --primary-hover: #1a1a1a;
    --secondary-color: #666666;
    --danger-color: #444444;
    --warning-color: #888888;
    --success-color: #22c55e;
    --bg-primary: #ffffff;
    --bg-secondary: #f9fafb;
    --bg-tertiary: #f3f4f6;
    --text-primary: #111827;
    --text-secondary: #6b7280;
    --text-tertiary: #9ca3af;
    --border-color: #e5e7eb;
    --border-radius: 12px;
    --shadow-sm: 0 1px 2px 0 rgba(0,0,0,.05);
    --shadow-md: 0 4px 6px -1px rgba(0,0,0,.1);
    --shadow-lg: 0 10px 15px -3px rgba(0,0,0,.1);
}
* { margin:0; padding:0; box-sizing:border-box; }
body { font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif; line-height:1.6; color:var(--text-primary); background:linear-gradient(135deg,#2a2a2a 0%,#1a1a1a 100%); min-height:100vh; padding:20px; }
.container { max-width:1200px; margin:0 auto; background:var(--bg-primary); border-radius:var(--border-radius); box-shadow:var(--shadow-lg); overflow:hidden; }
.header { background:linear-gradient(135deg,#2a2a2a 0%,#1a1a1a 100%); color:white; padding:40px 30px; text-align:center; }
.header h1 { font-size:2.5rem; margin-bottom:10px; font-weight:700; }
.subtitle { font-size:1.1rem; opacity:.9; margin-bottom:20px; }
.status-indicator { display:inline-flex; align-items:center; gap:8px; background:rgba(0,0,0,.2); padding:8px 16px; border-radius:20px; font-size:.9rem; }
.status-dot { width:10px; height:10px; border-radius:50%; background:#d1d5db; }
.status-dot.healthy { background:var(--success-color); animation:pulse 2s infinite; }
@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:.5} }
.main-content { padding:40px 30px; }
.upload-section { max-width:800px; margin:0 auto; }
.upload-area { border:3px dashed var(--border-color); border-radius:var(--border-radius); padding:60px 40px; text-align:center; background:var(--bg-secondary); transition:all .3s; cursor:pointer; position:relative; }
.upload-area:hover { border-color:var(--primary-color); background:var(--bg-tertiary); }
.upload-area.dragover { border-color:var(--primary-color); background:rgba(51,51,51,.1); transform:scale(1.02); }
.upload-icon { font-size:4rem; margin-bottom:20px; }
.upload-area h2 { font-size:1.5rem; margin-bottom:10px; }
.upload-area p { color:var(--text-secondary); }
.file-info { margin-top:20px; display:flex; align-items:center; justify-content:center; gap:10px; background:white; padding:15px 20px; border-radius:8px; border:1px solid var(--border-color); }
.file-name { font-weight:600; color:var(--primary-color); }
.btn-remove { background:var(--danger-color); color:white; border:none; border-radius:50%; width:24px; height:24px; cursor:pointer; display:flex; align-items:center; justify-content:center; font-size:1rem; transition:all .2s; }
.btn-remove:hover { transform:scale(1.1); }
.query-section { margin-top:30px; }
.query-section label { display:block; font-weight:600; margin-bottom:10px; }
.query-section textarea { width:100%; padding:15px; border:2px solid var(--border-color); border-radius:8px; font-family:inherit; font-size:1rem; resize:vertical; transition:border-color .3s; }
.query-section textarea:focus { outline:none; border-color:var(--primary-color); }
.example-queries { margin-top:15px; }
.example-queries p { font-size:.9rem; color:var(--text-secondary); margin-bottom:8px; }
.query-chips { display:flex; flex-wrap:wrap; gap:10px; }
.query-chip { padding:8px 16px; background:var(--bg-tertiary); border:1px solid var(--border-color); border-radius:20px; font-size:.85rem; cursor:pointer; transition:all .2s; }
.query-chip:hover { background:var(--primary-color); color:white; border-color:var(--primary-color); }
.btn-primary { width:100%; padding:16px 32px; background:var(--primary-color); color:white; border:none; border-radius:8px; font-size:1.1rem; font-weight:600; cursor:pointer; margin-top:30px; transition:all .3s; position:relative; }
.btn-primary:hover:not(:disabled) { background:var(--primary-hover); transform:translateY(-2px); box-shadow:var(--shadow-md); }
.btn-primary:disabled { background:var(--text-tertiary); cursor:not-allowed; }
.btn-loading { display:flex; align-items:center; justify-content:center; gap:10px; }
.spinner { width:20px; height:20px; border:3px solid rgba(255,255,255,.3); border-top-color:white; border-radius:50%; animation:spin 1s linear infinite; }
@keyframes spin { to{transform:rotate(360deg)} }
.btn-secondary { padding:10px 20px; background:white; color:var(--primary-color); border:2px solid var(--primary-color); border-radius:8px; font-weight:600; cursor:pointer; transition:all .2s; }
.btn-secondary:hover { background:var(--primary-color); color:white; }
.btn-download { padding:12px 24px; background:var(--bg-tertiary); border:2px solid var(--border-color); border-radius:8px; cursor:pointer; transition:all .2s; display:flex; align-items:center; gap:8px; }
.btn-download:hover { background:var(--primary-color); color:white; border-color:var(--primary-color); }
.results-section { animation:fadeIn .5s ease; }
@keyframes fadeIn { from{opacity:0;transform:translateY(20px)} to{opacity:1;transform:none} }
.results-header { display:flex; justify-content:space-between; align-items:center; margin-bottom:30px; }
.results-header h2 { font-size:2rem; }
.results-grid { display:grid; gap:20px; }
.result-card { background:white; border:1px solid var(--border-color); border-radius:var(--border-radius); padding:25px; box-shadow:var(--shadow-sm); }
.card-header { display:flex; justify-content:space-between; align-items:center; margin-bottom:20px; padding-bottom:15px; border-bottom:2px solid var(--bg-tertiary); }
.card-header h3 { font-size:1.3rem; }
.quality-badge { padding:6px 16px; border-radius:20px; font-size:.85rem; font-weight:600; }
.summary-text { white-space:normal; line-height:1.8; color:var(--text-primary); margin-bottom:20px; font-size:1rem; }
.metrics { display:grid; grid-template-columns:repeat(auto-fit,minmax(150px,1fr)); gap:15px; margin-top:20px; }
.metric { background:var(--bg-secondary); padding:15px; border-radius:8px; text-align:center; }
.metric-label { display:block; font-size:.85rem; color:var(--text-secondary); margin-bottom:5px; }
.metric-value { display:block; font-size:1.5rem; font-weight:700; color:var(--primary-color); }
.visualizations-grid { display:grid; grid-template-columns:repeat(auto-fit,minmax(300px,1fr)); gap:20px; }
.viz-item { border:1px solid var(--border-color); border-radius:8px; overflow:hidden; transition:transform .2s; }
.viz-item:hover { transform:scale(1.02); }
.viz-item img { width:100%; height:auto; display:block; }
.viz-label { padding:10px; background:var(--bg-tertiary); font-size:.9rem; color:var(--text-secondary); text-align:center; }
.steps-list { display:flex; flex-direction:column; gap:15px; }
.step-item { padding:15px; background:var(--bg-secondary); border-left:4px solid var(--primary-color); border-radius:4px; }
.step-item h4 { font-size:1rem; margin-bottom:5px; }
.step-item p { font-size:.9rem; color:var(--text-secondary); }
.download-section { margin-top:30px; padding:25px; background:var(--bg-secondary); border-radius:var(--border-radius); }
.download-section h3 { margin-bottom:15px; }
.download-buttons { display:flex; gap:15px; flex-wrap:wrap; }
.error-section { max-width:600px; margin:0 auto; }
.error-card { background:white; border:2px solid var(--danger-color); border-radius:var(--border-radius); padding:40px; text-align:center; }
.error-icon { font-size:4rem; margin-bottom:20px; }
.error-card h3 { color:var(--danger-color); margin-bottom:15px; }
.error-card p { color:var(--text-secondary); margin-bottom:25px; }
.footer { background:var(--bg-tertiary); padding:20px 30px; text-align:center; color:var(--text-secondary); font-size:.9rem; border-top:1px solid var(--border-color); }
.footer p { margin:5px 0; }
@media(max-width:768px) {
    body{padding:10px}
    .header h1{font-size:2rem}
    .main-content{padding:20px 15px}
    .upload-area{padding:40px 20px}
    .query-chips{flex-direction:column}
    .results-header{flex-direction:column;gap:15px;align-items:flex-start}
    .metrics{grid-template-columns:1fr}
    .download-buttons{flex-direction:column}
}
"""

# Write base CSS first, then append the extra styles
(ROOT / "static/css/styles.css").write_text(base_css)

css_extra = r"""
.apikey-section { max-width:600px; margin:0 auto 30px; }
.apikey-card { background:white; border:2px solid var(--border-color); border-radius:var(--border-radius); padding:30px; box-shadow:var(--shadow-sm); }
.apikey-card h2 { margin-bottom:10px; }
.apikey-card p { color:var(--text-secondary); margin-bottom:16px; font-size:.95rem; }
.apikey-card a { color:var(--primary-color); }
.apikey-input-row { display:flex; gap:10px; }
.apikey-input-row input { flex:1; padding:10px 14px; border:2px solid var(--border-color); border-radius:8px; font-size:1rem; font-family:monospace; }
.apikey-input-row input:focus { outline:none; border-color:var(--primary-color); }
.apikey-note { font-size:.8rem !important; color:var(--text-tertiary) !important; margin-top:10px !important; margin-bottom:0 !important; }
.change-key-row { text-align:center; margin-top:10px; }
.btn-link { background:none; border:none; color:var(--text-secondary); font-size:.85rem; cursor:pointer; text-decoration:underline; }
.btn-link:hover { color:var(--primary-color); }
.summary-text h1,.summary-text h2,.summary-text h3 { margin:12px 0 6px; color:var(--text-primary); }
.summary-text p { margin-bottom:10px; }
.summary-text ul,.summary-text ol { padding-left:20px; margin-bottom:10px; }
.summary-text li { margin-bottom:4px; }
.summary-text strong { font-weight:700; }
"""

with open(ROOT / "static/css/styles.css", "a") as f:
    f.write(css_extra)

print("✅ CSS written")

In [ ]:
# ── Cell 11: Start FastAPI server (background) ────────────────────────────────

import subprocess, time, requests

server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=str(ROOT)
)

# Wait until server is ready
for _ in range(30):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print("✅ Server is running on http://localhost:8000")
            break
    except Exception:
        pass
    time.sleep(2)
else:
    print("⚠️  Server may not have started — check output above")

In [ ]:
# ── Cell 12: Open public tunnel with cloudflared (no sign-in needed) ──────────

import subprocess, threading, re, time
from IPython.display import display, HTML

# Download cloudflared binary (one-time)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

public_url = None

def run_tunnel():
    global public_url
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break

t = threading.Thread(target=run_tunnel, daemon=True)
t.start()

# Wait for URL
for _ in range(30):
    if public_url:
        break
    time.sleep(2)

if public_url:
    display(HTML(f"""
    <div style="font-family:sans-serif;background:#f0fdf4;border:2px solid #22c55e;
                border-radius:12px;padding:24px;max-width:600px;margin:20px auto">
        <h2 style="color:#166534;margin-bottom:8px">&#127881; Your app is live!</h2>
        <p style="color:#166534;margin-bottom:16px">Share this link with anyone — no account or installation needed:</p>
        <a href="{public_url}" target="_blank"
           style="display:block;background:#166534;color:white;padding:14px 20px;
                  border-radius:8px;text-decoration:none;font-size:1.1rem;text-align:center;word-break:break-all">
            &#128279; {public_url}
        </a>
        <p style="color:#6b7280;font-size:.85rem;margin-top:12px">
            &#9888; This URL is temporary and changes when you restart the notebook.
            Keep this Colab tab open while the app is in use.
        </p>
    </div>
    """))
else:
    print("⚠️  Could not get a public URL. Try running this cell again.")